# 5G Network Intelligence — Week 4: GRU-A và GRU-B

## Cách đọc notebook

Notebook này dùng **code Python thật, có comment**, chia theo từng module.
Không có khối source base64. Mỗi cell `%%writefile` ghi đúng file được hiển thị
để bạn có thể sửa trực tiếp, chạy lại cell và debug module đó.

**Mặc định Run All chỉ review, không train lại.** Toàn bộ training chỉ được phép
trên hosted Colab/Kaggle. Kết quả hiện tại theo schema phase4 được retrain ở
[Week 5 GPU job](https://www.kaggle.com/code/truongbaokien/5g-network-intelligence-week5),
Version 1 → Output. Artifact static6 của Week 4 chỉ còn là evidence lịch sử.

### Mục lục
1. Kết quả đã đo và cấu hình chạy
2. Dữ liệu, schema và temporal split
3. Sampling, scaler/imputer và missingness masks
4. Kiến trúc GRU và vòng lặp training/resume
5. CLI orchestration và validation replay
6. Chạy từng bước, debug một beam và xem learning curves

**Luồng:** CSV → panel → training-only preprocessing → sequence 168h →
GRU + phase4 → forecast +1/+6/+24h → validation metrics.


## 1. Kết quả validation đã đo

MAE dùng đơn vị **DLPRB usage**. `skill > 0` nghĩa là cải thiện so với mô hình tham chiếu.
Mean ± sample SD của **2 seeds** chỉ mô tả biến động, không chứng minh statistical stability.
GRU tốt hơn best baseline nhưng **chưa vượt LightGBM C1**. Holdout chưa được đánh giá.


In [ ]:
# Bảng số đo thật từ reports/gru_comparison_val.csv; không chạy training ở cell này.
import pandas as pd
from IPython.display import display
MEASURED_RESULTS = [{'variant': 'A',
  'horizon': 1,
  'mae_mean': 0.2403396978205069,
  'mae_sd': 0.0005595959146663903,
  'skill_vs_baseline': 0.1928066300821316,
  'lightgbm_mae': 0.2306345114678315,
  'skill_vs_lightgbm': -0.042080373361768286},
 {'variant': 'A',
  'horizon': 6,
  'mae_mean': 0.27010271966169275,
  'mae_sd': 0.000687284351004755,
  'skill_vs_baseline': 0.18814640693943607,
  'lightgbm_mae': 0.2534728960180836,
  'skill_vs_lightgbm': -0.06560789695803493},
 {'variant': 'A',
  'horizon': 24,
  'mae_mean': 0.2622557697716986,
  'mae_sd': 0.0004351646847633241,
  'skill_vs_baseline': 0.21022874583696016,
  'lightgbm_mae': 0.2512464622398822,
  'skill_vs_lightgbm': -0.043818756426130756},
 {'variant': 'B',
  'horizon': 1,
  'mae_mean': 0.2388113410499222,
  'mae_sd': 0.00032260264152019614,
  'skill_vs_baseline': 0.19793969575239978,
  'lightgbm_mae': 0.2306345114678315,
  'skill_vs_lightgbm': -0.0354536254355462},
 {'variant': 'B',
  'horizon': 6,
  'mae_mean': 0.26454336262050515,
  'mae_sd': 0.005055475288482708,
  'skill_vs_baseline': 0.204856286775699,
  'lightgbm_mae': 0.2534728960180836,
  'skill_vs_lightgbm': -0.04367514939992545},
 {'variant': 'B',
  'horizon': 24,
  'mae_mean': 0.26098344120389727,
  'mae_sd': 0.0018581674681334768,
  'skill_vs_baseline': 0.21406030511809504,
  'lightgbm_mae': 0.2512464622398822,
  'skill_vs_lightgbm': -0.038754690821152904}]
overview = pd.DataFrame(MEASURED_RESULTS)
display(overview.round(6))


variant,horizon,mae_mean,mae_sd,skill_vs_baseline,lightgbm_mae,skill_vs_lightgbm
A,1,0.240340,0.000560,0.192807,0.230635,-0.042080
A,6,0.270103,0.000687,0.188146,0.253473,-0.065608
A,24,0.262256,0.000435,0.210229,0.251246,-0.043819
B,1,0.238811,0.000323,0.197940,0.230635,-0.035454
B,6,0.264543,0.005055,0.204856,0.253473,-0.043675
B,24,0.260983,0.001858,0.214060,0.251246,-0.038755


### Cấu hình và các cờ thực thi

Để review/debug source: giữ các cờ bên dưới là `False`. Khi muốn chạy lại, bật từng
bước và chọn GPU trong Settings. **Không bật RUN_TRAINING khi chỉ muốn đọc notebook.**
Chọn một variant/seed để debug trước khi chạy bốn experiments. Artifact có sẵn cần
`RESUME=True`; chương trình không ghi đè một run đã có checkpoint.


In [ ]:
BUILD_SOURCE_COMMIT = '5c912d1576b1e9ce3b54ebc8cb7744e2c529123f'
from pathlib import Path
import os
import sys
import subprocess

# Đây là runtime cloud, không phải máy cá nhân. Không có training ở cell khởi tạo.
IS_KAGGLE = Path('/kaggle/working').is_dir()
IS_COLAB = not IS_KAGGLE and 'COLAB_RELEASE_TAG' in os.environ
if not (IS_KAGGLE or IS_COLAB):
    raise RuntimeError('Chỉ chạy notebook này trong hosted Kaggle hoặc Colab.')
PROJECT = Path('/kaggle/working/5g-review' if IS_KAGGLE else '/content/5g-review')
PROJECT.mkdir(exist_ok=True)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
os.environ['SOURCE_GIT_COMMIT'] = BUILD_SOURCE_COMMIT
for folder in ('src/utils', 'src/data', 'src/evaluation', 'src/features', 'src/models', 'scripts', 'configs', 'reports'):
    path = PROJECT / folder
    path.mkdir(parents=True, exist_ok=True)
    if not folder.startswith(('configs', 'reports')):
        (path / '__init__.py').touch()
(PROJECT / 'src/__init__.py').touch()

RUN_INSTALL = False
RUN_PREPARE_DATA = False
RUN_TRAINING = False
RUN_ALL_EXPERIMENTS = False
RUN_EVALUATION = False
DEBUG_DATA = False
VARIANT = 'B'  # A: DLPRB + mask; B: 4 KPI + 4 masks
SEED = 17
RESUME = False
DEBUG_ORIGIN = 503  # first fallback validation origin
DEBUG_BEAM = 0
print('Project:', PROJECT)


## 2. Dữ liệu và temporal split

Time là relative hourly index; split fallback: fit 0–503, validation 504–671.
Không dùng holdout 672–839 để tune. Các file dưới đây kiểm tra schema, checksum và boundary.


### `src/utils/config.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/utils/config.py
"""Configuration loading."""
from pathlib import Path

import yaml


def load_config(path: str | Path) -> dict:
    """Load a YAML mapping; reject empty or non-mapping configuration."""
    with Path(path).open(encoding="utf-8") as handle:
        config = yaml.safe_load(handle)
    if not isinstance(config, dict):
        raise ValueError("Configuration must be a mapping")
    return config


### `src/utils/seed.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/utils/seed.py
"""Deterministic random seeds."""
import random

import numpy as np


def set_seed(seed: int) -> None:
    """Seed Python and NumPy; neural training seeds belong to its runtime."""
    random.seed(seed)
    np.random.seed(seed)


### `src/utils/metadata.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/utils/metadata.py
"""Run provenance without machine-specific paths or secrets."""
import hashlib
import json
import os
import platform
import subprocess
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path


def collect_run_metadata(config: dict) -> dict:
    """Return config hash, environment, seed, revision and current Git commit."""
    result = subprocess.run(
        ["git", "-c", f"safe.directory={Path.cwd().as_posix()}", "rev-parse", "HEAD"],
        capture_output=True, text=True, check=False)
    return {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "config": config,
        "config_sha256": hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest(),
        "git_commit": result.stdout.strip() if result.returncode == 0 else os.environ.get("SOURCE_GIT_COMMIT"),
        "source_bundle_sha256": os.environ.get("SOURCE_BUNDLE_SHA256"),
        "python": platform.python_version(),
        "packages": {name: version(name) for name in ("numpy", "pandas", "PyYAML")},
        "seed": config.get("seed"),
        "dataset_revision": config.get("revision"),
    }


### `src/utils/runtime.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/utils/runtime.py
"""Owner policy: all model fitting must happen in hosted Colab or Kaggle."""
import os
from pathlib import Path


def is_cloud_runtime() -> bool:
    """Recognize hosted notebook runtime markers; ordinary local runs are rejected."""
    if os.name == "nt":
        return False
    return (Path("/kaggle/working").is_dir()
            or ("COLAB_RELEASE_TAG" in os.environ and Path("/content").is_dir()))


def require_cloud_training() -> None:
    """Refuse model training outside the owner's allowed hosted notebook runtimes."""
    if not is_cloud_runtime():
        raise RuntimeError("Local model training is forbidden by the owner. Use Colab or Kaggle.")


### `src/data/schema.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/data/schema.py
"""Canonical KPI and numeric beam schema."""
import re

KPIS = ("DLPRB", "DLThpVol", "DLThpTime", "MR_number")
SEGMENTS = {"train": ("train", "0w-5w"), "week6": ("test", "5w-6w"),
            "week11": ("test", "10w-11w")}


def parse_beam_id(beam_id: str) -> tuple[int, int, int]:
    """Parse a strict bs_cell_beam identifier into numeric components."""
    if not re.fullmatch(r"\d+_\d+_\d+", beam_id):
        raise ValueError(f"Invalid beam ID: {beam_id!r}")
    return tuple(int(part) for part in beam_id.split("_"))


def csv_path(kpi: str, segment: str) -> str:
    """Return repository-relative CSV path for one KPI and segment."""
    folder, period = SEGMENTS[segment]
    return f"data/{folder}/{kpi}_{folder}_{period}.csv"


### `src/data/io.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/data/io.py
"""Revision-pinned downloads and checksummed CSV manifests."""
import csv
import hashlib
import json
from pathlib import Path

from src.data.schema import KPIS, SEGMENTS, csv_path


def sha256(path: Path) -> str:
    """Compute a streaming file digest."""
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def download_dataset(repo_id: str, revision: str, dest: str | Path) -> dict:
    """Resolve revision once and download required CSVs; verify cached checksums."""
    from huggingface_hub import HfApi, hf_hub_download

    dest = Path(dest)
    dest.mkdir(parents=True, exist_ok=True)
    info = HfApi().dataset_info(repo_id, revision=revision)
    provenance_path = dest / "provenance.json"
    previous = json.loads(provenance_path.read_text()) if provenance_path.exists() else {}
    checksums = {}
    for segment in SEGMENTS:
        for kpi in KPIS:
            name = csv_path(kpi, segment)
            target = dest / name
            if (previous.get("revision") == info.sha and target.exists()
                    and previous.get("checksums", {}).get(name) == sha256(target)):
                checksums[name] = sha256(target)
                continue
            hf_hub_download(repo_id, name, repo_type="dataset", revision=info.sha,
                            local_dir=dest, force_download=(
                                previous.get("revision") == info.sha and target.exists()))
            checksums[name] = sha256(target)
    hf_hub_download(repo_id, "README.md", repo_type="dataset", revision=info.sha, local_dir=dest)
    provenance = {"repo_id": repo_id, "revision": info.sha,
                  "license": info.card_data.to_dict().get("license", "unknown")
                  if info.card_data else "unknown",
                  "checksums": checksums}
    provenance_path.write_text(json.dumps(provenance, indent=2), encoding="utf-8")
    return provenance


def build_manifest(raw_dir: str | Path) -> dict:
    """Inspect CSV row counts, columns, byte sizes and checksums."""
    raw_dir = Path(raw_dir)
    provenance = json.loads((raw_dir / "provenance.json").read_text(encoding="utf-8"))
    expected = set(provenance.get("checksums", {}))
    if not expected:
        raise ValueError("Provenance must include source file checksums")
    actual = {p.relative_to(raw_dir).as_posix() for p in raw_dir.rglob("*.csv")}
    if actual != expected:
        raise ValueError(f"Source files differ: missing={sorted(expected - actual)}, "
                         f"unexpected={sorted(actual - expected)}")
    files = []
    for path in sorted(raw_dir.rglob("*.csv")):
        with path.open(newline="", encoding="utf-8-sig") as handle:
            reader = csv.reader(handle)
            columns = next(reader)
            rows = sum(1 for _ in reader)
        digest = sha256(path)
        relative = path.relative_to(raw_dir).as_posix()
        if provenance["checksums"].get(relative) != digest:
            raise ValueError(f"Checksum mismatch: {relative}")
        files.append({"path": relative, "sha256": digest, "size_bytes": path.stat().st_size,
                      "rows": rows, "columns": columns})
    return {**provenance, "files": files}


### `src/data/panel.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/data/panel.py
"""Strict, aligned time × beam × KPI arrays with explicit missing observations."""
import csv
import json
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd

from src.data.schema import KPIS, parse_beam_id


@dataclass
class Panel:
    """Aligned values with source-relative time indices and provenance."""

    values: np.ndarray
    time: np.ndarray
    beam_ids: tuple[str, ...]
    metadata: dict

    def __post_init__(self) -> None:
        if self.values.shape != (len(self.time), len(self.beam_ids), len(KPIS)):
            raise ValueError("Panel shape must match time, beam and KPI axes")
        if len(set(self.beam_ids)) != len(self.beam_ids):
            raise ValueError("Duplicate beam identifiers")
        if not np.issubdtype(self.time.dtype, np.integer) or np.any(np.diff(self.time) <= 0):
            raise ValueError("Time must contain strictly increasing integer indices")
        if np.isinf(self.values).any():
            raise ValueError("Infinite KPI values are not supported")


def build_panel(files: dict[str, str | Path], metadata: dict | None = None) -> Panel:
    """Outer-align four wide CSVs; reject duplicate keys and malformed numbers."""
    if set(files) != set(KPIS):
        raise ValueError(f"Exactly these KPIs are required: {KPIS}")
    frames = {}
    source_audit = {}
    for kpi in KPIS:
        path = Path(files[kpi])
        with path.open(encoding="utf-8-sig", newline="") as handle:
            header = next(csv.reader(handle))
        # Actual release uses an unnamed first index column rather than card's Time.
        unnamed_index = bool(header and header[0] == "" and "Time" not in header)
        if unnamed_index:
            header[0] = "Time"
        if len(header) != len(set(header)) or header.count("Time") != 1:
            raise ValueError(f"Duplicate columns or missing Time: {path}")
        frame = pd.read_csv(path)
        if unnamed_index:
            frame = frame.rename(columns={frame.columns[0]: "Time"})
        frame = frame.set_index("Time")
        numeric_time = pd.to_numeric(frame.index, errors="raise").to_numpy(dtype=float)
        if not np.isfinite(numeric_time).all() or np.any(numeric_time % 1):
            raise ValueError("Time must be finite integer indices")
        frame.index = numeric_time.astype(np.int64)
        if frame.index.has_duplicates:
            raise ValueError(f"Duplicate Time keys: {path}")
        for beam in frame.columns:
            parse_beam_id(beam)
        if not len(frame) or not len(frame.columns):
            raise ValueError(f"Empty KPI file: {path}")
        frame = frame.apply(pd.to_numeric, errors="raise")
        frames[kpi] = frame
        source_audit[kpi] = {"rows": len(frame), "beams": len(frame.columns),
                             "unnamed_time_column": unnamed_index,
                             "duplicate_time_keys": 0, "duplicate_beam_columns": 0}
    beams = tuple(sorted(set().union(*(f.columns for f in frames.values())), key=parse_beam_id))
    time = np.array(sorted(set().union(*(f.index for f in frames.values()))), dtype=np.int64)
    values = np.stack([frames[k].reindex(index=time, columns=beams).to_numpy(dtype=np.float64)
                       for k in KPIS], axis=-1)
    return Panel(values, time, beams, {**(metadata or {}), "source_audit": source_audit,
                                      "kpis": list(KPIS)})


def save_panel(panel: Panel, path: str | Path) -> None:
    """Save a pickle-free NPZ and adjacent human-readable metadata JSON."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(path, values=panel.values, time=panel.time,
                        beam_ids=np.array(panel.beam_ids),
                        metadata=np.array(json.dumps(panel.metadata)))
    path.with_name(f"{path.stem}_meta.json").write_text(
        json.dumps({**panel.metadata, "shape": list(panel.values.shape)}, indent=2),
        encoding="utf-8")


def load_panel(path: str | Path) -> Panel:
    """Load and validate a saved panel without pickle."""
    with np.load(path, allow_pickle=False) as data:
        return Panel(data["values"], data["time"], tuple(data["beam_ids"].tolist()),
                     json.loads(str(data["metadata"])))


### `src/data/audit.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/data/audit.py
"""Descriptive data audits; no imputation, filtering or model fitting."""
import numpy as np

from src.data.panel import Panel
from src.data.schema import KPIS, parse_beam_id


def audit_missing(panel: Panel) -> dict:
    """Return missing ratios per KPI and per beam."""
    mask = np.isnan(panel.values)
    return {"per_kpi": dict(zip(KPIS, mask.mean(axis=(0, 1)).tolist())),
            "per_beam": mask.mean(axis=(0, 2)).tolist()}


def audit_zeros(panel: Panel) -> dict:
    """Return zero/active ratios using observed values as denominator."""
    result = {}
    for i, kpi in enumerate(KPIS):
        values = panel.values[:, :, i]
        observed = values[~np.isnan(values)]
        result[kpi] = {"observed": len(observed),
                       "zero_ratio": float(np.mean(observed == 0)) if observed.size else None,
                       "active_ratio": float(np.mean(observed > 0)) if observed.size else None}
    return result


def audit_range(panel: Panel) -> dict:
    """Report finite range, quantiles and negative counts without deleting values."""
    result = {}
    for i, kpi in enumerate(KPIS):
        values = panel.values[:, :, i]
        observed = values[np.isfinite(values)]
        result[kpi] = {"negative_count": int((observed < 0).sum()),
                       "quantiles": np.quantile(observed, [0, .5, .9, .95, 1]).tolist()
                       if observed.size else None}
    return result


def audit_duplicates(panel: Panel) -> dict:
    """Report key duplicates; builder rejects these before alignment."""
    return {"duplicate_time_keys": len(panel.time) - len(np.unique(panel.time)),
            "duplicate_beam_ids": len(panel.beam_ids) - len(set(panel.beam_ids)),
            "source_files": panel.metadata.get("source_audit", {})}


def check_mapping(panel: Panel) -> dict:
    """Check identifiers against the documented 30 × 3 × 32 network topology."""
    ids = [parse_beam_id(beam) for beam in panel.beam_ids]
    expected = {(bs, cell, beam) for bs in range(30) for cell in range(3) for beam in range(32)}
    return {"complete_expected_mapping": set(ids) == expected,
            "within_expected_bounds": all(item in expected for item in ids),
            "base_stations": len({item[0] for item in ids}),
            "cells": len({item[:2] for item in ids}), "beams": len(ids)}


def audit_beams(panel: Panel) -> dict:
    """Return topology counts and all-missing/constant beam counts."""
    target = panel.values[:, :, 0]
    constant = sum(len(np.unique(col[np.isfinite(col)])) == 1 for col in target.T)
    return {**check_mapping(panel), "all_missing_target_beams": int(np.isnan(target).all(0).sum()),
            "constant_target_beams": constant}


def check_temporal_boundary(train: Panel, week6: Panel, *, verified: bool = False,
                            evidence: str = "") -> dict:
    """Require documented continuity and exact source indices before concatenation."""
    structure_ok = (np.array_equal(train.time, np.arange(840))
                    and np.array_equal(week6.time, np.arange(168))
                    and train.beam_ids == week6.beam_ids)
    allowed = bool(verified and evidence.strip() and structure_ok)
    return {"continuity_verified": allowed, "structure_ok": bool(structure_ok),
            "evidence": evidence, "week6_time_offset": 840 if allowed else None,
            "reason": "Documented continuity and indices verified" if allowed else
            "Use internal holdout: continuity has not passed evidence and structure checks"}


### `src/evaluation/origins.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/evaluation/origins.py
"""Integer forecast origins; all split endpoints are inclusive."""
import numpy as np


def _validate(horizon: int, lookback: int, stride: int = 1) -> None:
    if any(not isinstance(x, (int, np.integer)) or isinstance(x, bool) or x < 1
           for x in (horizon, lookback, stride)):
        raise ValueError("horizon, lookback and stride must be positive integers")


def train_origins(fit_end: int, horizon: int, lookback: int = 168,
                  stride: int = 1) -> np.ndarray:
    """Return origins whose full history and target lie in the fit segment."""
    _validate(horizon, lookback, stride)
    return np.arange(lookback - 1, fit_end - horizon + 1, stride, dtype=np.int64)


def evaluation_origins(fit_end: int, segment_end: int, horizon: int,
                       lookback: int = 168) -> np.ndarray:
    """Use fixed-model rolling origins from fit_end, including observed evaluation history."""
    _validate(horizon, lookback)
    if segment_end <= fit_end:
        raise ValueError("Evaluation segment must follow fitted data")
    return np.arange(max(lookback - 1, fit_end), segment_end - horizon + 1,
                     dtype=np.int64)


def train_origins_joint(fit_end: int, horizons: tuple[int, ...] = (1, 6, 24),
                        lookback: int = 168, stride: int = 1) -> np.ndarray:
    """Return training origins that have all requested horizon targets."""
    if not horizons:
        raise ValueError("At least one horizon is required")
    for horizon in horizons:
        _validate(horizon, lookback, stride)
    return train_origins(fit_end, max(horizons), lookback, stride)


def assert_no_target_overlap(segments: dict[str, dict[int, np.ndarray]]) -> None:
    """Reject target overlap between segments, considering every horizon together."""
    seen: set[int] = set()
    for name, horizon_origins in segments.items():
        targets = {int(origin + h) for h, origins in horizon_origins.items() for origin in origins}
        if seen & targets:
            raise ValueError(f"Target overlap at segment {name}: {sorted(seen & targets)[:5]}")
        seen |= targets


def split_origins(split: dict) -> dict[str, dict[int, np.ndarray]]:
    """Build train/validation/holdout origin sets without reading any observations."""
    lookback = split["lookback"]
    result = {
        "train": {h: train_origins(split["fit"][1], h, lookback) for h in split["horizons"]},
        "validation": {h: evaluation_origins(split["fit"][1], split["validation"][1], h,
                                              lookback) for h in split["horizons"]},
        "holdout": {h: evaluation_origins(split["final_refit"][1], split["holdout"][1], h,
                                           lookback) for h in split["horizons"]},
    }
    assert_no_target_overlap(result)
    return result


### `src/data/splits.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/data/splits.py
"""Frozen chronological split definitions."""


def build_split_config(n_hours: int, continuity_verified: bool = False) -> dict:
    """Return inclusive split bounds; reject lengths inconsistent with selected protocol."""
    expected = 1008 if continuity_verified else 840
    if n_hours != expected:
        raise ValueError(f"Expected {expected} hourly rows; received {n_hours}")
    fit_end = 671 if continuity_verified else 503
    split = {"mode": "primary" if continuity_verified else "fallback",
            "bounds": "inclusive", "lookback": 168, "horizons": [1, 6, 24],
            "fit": [0, fit_end], "validation": [fit_end + 1, fit_end + 168],
            "holdout": [fit_end + 169, fit_end + 336],
            "final_refit": [0, fit_end + 168], "week11": "excluded"}
    from src.evaluation.origins import split_origins

    split["origins"] = {segment: {str(h): origins.tolist() for h, origins in horizons.items()}
                        for segment, horizons in split_origins(split).items()}
    return split


### `src/evaluation/validation.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/evaluation/validation.py
"""Shared validation-only data boundary checks for training and replay."""
import json
from pathlib import Path

from src.data.panel import Panel, load_panel
from src.data.splits import build_split_config
from src.utils.config import load_config


def validation_inputs(config: dict) -> tuple[Panel, dict]:
    """Validate frozen split/config and return a panel truncated before holdout."""
    processed = Path(config["processed_dir"])
    panel = load_panel(processed / "panel.npz")
    split = json.loads((processed / "split.json").read_text())
    if split["mode"] not in ("primary", "fallback"):
        raise ValueError("Unknown frozen split mode")
    expected = build_split_config(len(panel.time), split["mode"] == "primary")
    evaluation = load_config(config["evaluation_config"])
    for key in ("fit", "validation", "holdout", "lookback", "horizons", "final_refit"):
        if split[key] != expected[key]:
            raise ValueError(f"Unexpected split boundary: {key}")
        if key != "final_refit" and evaluation[key] != split[key]:
            raise ValueError(f"Evaluation config disagrees: {key}")
    if evaluation["split_mode"] != split["mode"]:
        raise ValueError("Evaluation mode disagrees")
    end = split["validation"][1] + 1
    return Panel(panel.values[:end], panel.time[:end], panel.beam_ids, panel.metadata), expected


## 3. Sampling và GRU inputs

`compute_static`: chỉ fit prefix training. `sample_beams`: cùng 600 beams cho A/B và seeds.
`GRUPreprocessor`: median imputer + scaler training-only; mask 1 là missing, zero vẫn là observation.
`GRUBatches`: lấy đúng t−167..t, target t+[1,6,24]; không materialize toàn bộ sliding windows.


### `src/features/static_stats.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/features/static_stats.py
"""Beam statistics fitted on a declared training prefix only."""
from dataclasses import dataclass

import numpy as np

from src.data.panel import Panel


@dataclass
class StaticStats:
    """Per-beam mean, population std, P90 and observed-only active ratio."""

    values: np.ndarray
    beam_ids: tuple[str, ...]
    fit_end: int


def compute_static(panel: Panel, fit_end: int) -> StaticStats:
    """Compute statistics from indices 0..fit_end; all-missing beams stay NaN."""
    if not isinstance(fit_end, int) or not 0 <= fit_end < len(panel.time):
        raise ValueError("fit_end must identify a training prefix")
    target = panel.values[:fit_end + 1, :, 0]
    result = np.full((len(panel.beam_ids), 4), np.nan)
    for i, column in enumerate(target.T):
        observed = column[np.isfinite(column)]
        if observed.size:
            result[i] = [observed.mean(), observed.std(ddof=0), np.quantile(observed, .9),
                         np.mean(observed > 0)]
    return StaticStats(result, panel.beam_ids, fit_end)


### `src/features/beam_sampling.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/features/beam_sampling.py
"""Reproducible stratified beam sampling from training-only statistics."""
import numpy as np

from src.features.static_stats import StaticStats


def sample_beams(static: StaticStats, size: int = 600, seed: int = 17) -> np.ndarray:
    """Sample proportionally across joint tertiles of mean/std/active ratio."""
    if not isinstance(size, int) or size < 1:
        raise ValueError("sample size must be positive")
    valid = np.flatnonzero(np.isfinite(static.values).all(1))
    if not valid.size:
        raise ValueError("No beams have observed fit targets")
    size = min(size, len(valid))
    features = static.values[valid][:, [0, 1, 3]]
    bins = np.stack([np.searchsorted(np.quantile(features[:, k], [1/3, 2/3]),
                                     features[:, k], side="left") for k in range(3)], axis=1)
    strata = bins @ np.array([9, 3, 1])
    labels, counts = np.unique(strata, return_counts=True)
    desired = counts * size / counts.sum()
    # Allocate integer quotas proportionally, then distribute leftover slots by largest remainder.
    allocations = np.floor(desired).astype(int)
    for i in np.argsort(-(desired - allocations), kind="stable")[:size - allocations.sum()]:
        allocations[i] += 1
    rng = np.random.default_rng(seed)
    selected = [rng.choice(valid[strata == label], count, replace=False)
                for label, count in zip(labels, allocations) if count]
    return np.sort(np.concatenate(selected))


### `src/features/gru_inputs.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/features/gru_inputs.py
"""Training-fitted imputation/scaling and causal GRU batches without window materialization."""
from dataclasses import dataclass
from pathlib import Path
from collections.abc import Iterator

import numpy as np
import torch

from src.data.panel import Panel

SCHEMA = "gru-global-kpi-phase4-v2"


def fit_imputer(panel: Panel, fit_end: int, variant: str = "B") -> np.ndarray:
    """Fit one global median per selected KPI, with no beam dimension."""
    if variant not in ("A", "B") or not 0 <= fit_end < len(panel.time):
        raise ValueError("Invalid variant or fit cutoff")
    result = []
    for k in range(1 if variant == "A" else 4):
        values = panel.values[:fit_end+1, :, k].reshape(-1)
        finite = values[np.isfinite(values)]
        result.append(float(np.median(finite)) if finite.size else 0.)
    return np.asarray(result)


def fit_scaler(panel: Panel, fit_end: int, median: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Fit global per-KPI mean/std after training-median imputation, not per beam."""
    if not 0 <= fit_end < len(panel.time) or median.shape not in ((1,), (4,)):
        raise ValueError("Invalid cutoff or KPI parameter shape")
    means, scales = [], []
    for k, fill in enumerate(median):
        values = panel.values[:fit_end+1, :, k].reshape(-1)
        imputed = np.where(np.isfinite(values), values, fill)
        means.append(float(imputed.mean()))
        spread = float(imputed.std(ddof=0))
        scales.append(spread if spread >= 1e-8 else 1.)
    return np.asarray(means), np.asarray(scales)


@dataclass
class GRUPreprocessor:
    """Global per-KPI parameters; beam IDs are metadata, never fitted beam parameters."""

    median: np.ndarray
    mean: np.ndarray
    scale: np.ndarray
    beam_ids: tuple[str, ...]
    fit_end: int
    variant: str = "B"

    @classmethod
    def fit(cls, panel: Panel, fit_end: int, variant: str = "B") -> "GRUPreprocessor":
        """Fit only the training prefix; entirely missing KPIs use zero/one fallback."""
        median = fit_imputer(panel, fit_end, variant)
        mean, scale = fit_scaler(panel, fit_end, median)
        return cls(median, mean, scale, panel.beam_ids, fit_end, variant)

    def transform(self, values: np.ndarray, variant: str) -> np.ndarray:
        """Return scaled KPI channels followed by 1=missing masks (A:2, B:8 channels)."""
        if variant != self.variant or values.shape[-1] != 4:
            raise ValueError("Expected variant A/B and four KPI channels")
        values = values[..., :len(self.mean)]
        missing = ~np.isfinite(values)
        # Zero is observed. Only nonfinite entries are replaced; masks retain that information.
        scaled = (np.where(missing, self.median, values) - self.mean) / self.scale
        return np.concatenate([scaled, missing], axis=-1).astype(np.float32)

    def static_at(self, origins: np.ndarray, beams: np.ndarray) -> np.ndarray:
        """Return only sin/cos phases; beam identity/statistics cannot change these four values."""
        columns = []
        for period in (24, 168):
            angle = 2 * np.pi * origins / period
            columns.append(np.stack([np.sin(angle), np.cos(angle)], axis=-1))
        return np.concatenate(columns, axis=-1).astype(np.float32)

    def save(self, path: str | Path) -> None:
        """Persist numerical preprocessing state without pickle."""
        np.savez_compressed(path, median=self.median, mean=self.mean, scale=self.scale,
                            beam_ids=np.array(self.beam_ids), fit_end=self.fit_end,
                            variant=np.array(self.variant), schema=np.array(SCHEMA))

    @classmethod
    def load(cls, path: str | Path) -> "GRUPreprocessor":
        """Restore a trusted schema from a pickle-free numerical artifact."""
        with np.load(path, allow_pickle=False) as data:
            if "schema" not in data or str(data["schema"]) != SCHEMA or "static" in data:
                raise ValueError("Legacy/incompatible preprocessing artifact: retrain required")
            variant = str(data["variant"])
            expected = 1 if variant == "A" else 4
            if any(data[k].shape != (expected,) for k in ("median", "mean", "scale")):
                raise ValueError("Preprocessing parameters must be global per-KPI vectors")
            return cls(data["median"], data["mean"], data["scale"],
                       tuple(data["beam_ids"].tolist()), int(data["fit_end"]), variant)


class GRUBatches:
    """Keep compact panel tensors on the selected device and assemble windows per batch."""

    def __init__(self, panel: Panel, preprocessor: GRUPreprocessor, variant: str,
                 origins: np.ndarray, beam_indices: np.ndarray, device: str = "cpu") -> None:
        if panel.beam_ids != preprocessor.beam_ids:
            raise ValueError("Preprocessor beam order differs")
        self.origins, self.beams = np.asarray(origins), np.asarray(beam_indices)
        if (not self.origins.size or self.origins.min() < 167
                or self.origins.max() + 24 >= len(panel.time)):
            raise ValueError("Joint origins require 168 history rows and all three future targets")
        self.device = device
        self.values = torch.as_tensor(preprocessor.transform(panel.values, variant), device=device)
        self.target = torch.as_tensor(((panel.values[:, :, 0] - preprocessor.mean[0]) /
                                       preprocessor.scale[0]).astype(np.float32), device=device)
        self._preprocessor = preprocessor
        self.pairs = np.array(np.meshgrid(self.origins, self.beams, indexing="ij")).reshape(2, -1).T
        self.n_samples = len(self.pairs)

    def batches(self, batch_size: int, shuffle_seed: int | None = None
                ) -> Iterator[tuple[torch.Tensor, torch.Tensor, torch.Tensor]]:
        """Yield (history, phase4, target3) tensors; NaN labels remain masked."""
        order = np.arange(self.n_samples)
        if shuffle_seed is not None:
            np.random.default_rng(shuffle_seed).shuffle(order)
        offsets = torch.arange(-167, 1, device=self.device)
        horizons = torch.tensor([1, 6, 24], device=self.device)
        for start in range(0, self.n_samples, batch_size):
            pairs = self.pairs[order[start:start+batch_size]]
            origins = torch.as_tensor(pairs[:, 0], device=self.device)
            beams = torch.as_tensor(pairs[:, 1], device=self.device)
            # For origin t, index exactly t-167..t. No target/future value enters the sequence.
            sequence = self.values[origins[:, None] + offsets, beams[:, None]]
            static = torch.as_tensor(self._preprocessor.static_at(pairs[:, 0], pairs[:, 1]),
                                     device=self.device)
            target = self.target[origins[:, None] + horizons, beams[:, None]]
            yield sequence, static, target


## 4. GRU architecture và training loop

Input A có 2 channels, B có 8. GRU hidden=64, layers=2, dropout=0.1.
Phase4 = sin/cos phase 24h/168h; no beam statistics enter model inputs. Head trả 3 targets.
Loss chỉ lấy finite labels; AdamW + gradient clipping; validation early stopping.
`last.pt` phục vụ resume (optimizer/RNG/patience); `best.pt` phục vụ benchmark.


### `src/models/gru.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/models/gru.py
"""Limited GRU-A/B forecasting architecture from the execution plan."""
import torch
from torch import nn


class GRUNet(nn.Module):
    """Encode history and concatenate exactly four deterministic phase features."""

    def __init__(self, input_size: int, hidden_size: int = 64, num_layers: int = 2,
                 dropout: float = .1) -> None:
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                           dropout=dropout if num_layers > 1 else 0)
        self.head = nn.Sequential(nn.Linear(hidden_size+4, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, sequence: torch.Tensor, static: torch.Tensor) -> torch.Tensor:
        """Map (batch,time,channels) and (batch,4 phase) to (batch,3)."""
        _, hidden = self.gru(sequence)
        return self.head(torch.cat([hidden[-1], static], dim=1))


### `src/models/gru_train.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/models/gru_train.py
"""Masked L1 training with resumable optimizer/RNG state and per-epoch checkpoints."""
import json
import os
import time
from pathlib import Path

import torch

from src.features.gru_inputs import GRUBatches
from src.models.gru import GRUNet
from src.utils.runtime import require_cloud_training


def masked_l1(predicted: torch.Tensor, actual: torch.Tensor) -> tuple[torch.Tensor, int]:
    """Return mean absolute error among finite targets and number of valid targets."""
    mask = torch.isfinite(actual)
    # Exclude missing labels from supervision; do not impute the forecast target.
    count = int(mask.sum())
    if not count:
        return predicted.sum() * 0, 0
    return torch.abs(predicted[mask] - actual[mask]).mean(), count


def _save(state: dict, path: Path) -> None:
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    os.replace(temporary, path)


def train_gru(model: GRUNet, train: GRUBatches, validation: GRUBatches, config: dict,
               directory: str | Path, seed: int, resume: bool = False,
               allow_cpu_test: bool = False, run_identity: dict | None = None,
               seconds_limit: float | None = None) -> dict:
    """Train with AdamW, clipping, early stopping; reject full training on CPU."""
    require_cloud_training()
    if train.device == "cpu" and not allow_cpu_test:
        raise ValueError("Full GRU training requires GPU; CPU is allowed only for small tests")
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    model.to(train.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"])
    start, best, best_epoch, stale = 0, float("inf"), -1, 0
    ignored = {"max_epochs", "variants", "seeds", "artifact_dir", "gpu_budget_seconds"}
    identity = {"seed": seed, "config": {k: v for k, v in config.items() if k not in ignored},
                "run_identity": run_identity}
    if resume:
        state = torch.load(directory / "last.pt", map_location=train.device, weights_only=True)
        state["identity"]["config"] = {k: v for k, v in state["identity"]["config"].items()
                                           if k not in ignored}
        if state["identity"] != identity:
            raise ValueError("Resume identity differs from checkpoint")
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        torch.set_rng_state(state["rng"].cpu())
        if train.device != "cpu":
            torch.cuda.set_rng_state_all([x.cpu() for x in state["cuda_rng"]])
        start, best, best_epoch, stale = state["epoch"]+1, state["best"], state["best_epoch"], state["stale"]
    completed = start
    run_started = time.perf_counter()
    budget_exhausted = False
    for epoch in range(start, config["max_epochs"]):
        if stale >= config["early_stopping_patience"]:
            break
        began = time.perf_counter()
        model.train()
        total, count = 0., 0
        for sequence, static, target in train.batches(config["batch_size"], seed + epoch):
            optimizer.zero_grad(set_to_none=True)
            loss, n = masked_l1(model(sequence, static), target)
            if not n:
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config["gradient_clip"])
            optimizer.step()
            total += float(loss.detach()) * n
            count += n
        model.eval()
        val_total, val_count = 0., 0
        with torch.no_grad():
            for sequence, static, target in validation.batches(config["batch_size"]):
                loss, n = masked_l1(model(sequence, static), target)
                val_total += float(loss) * n
                val_count += n
        if not count or not val_count:
            raise ValueError("No observed training/validation targets")
        score = val_total / val_count
        # Selection uses validation loss only. Save the best model separately from resume state.
        improved = score < best
        best, best_epoch, stale = (score, epoch, 0) if improved else (best, best_epoch, stale + 1)
        state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(), "epoch": epoch,
                 "best": best, "best_epoch": best_epoch, "stale": stale, "identity": identity,
                 "rng": torch.get_rng_state(), "cuda_rng": torch.cuda.get_rng_state_all()
                 if train.device != "cpu" else []}
        _save(state, directory / "last.pt")
        if improved:
            _save(state, directory / "best.pt")
        row = {"epoch": epoch, "train_l1_scaled": total/count, "val_l1_scaled": score,
               "seconds": time.perf_counter()-began, "best_epoch": best_epoch,
               "train_valid_targets": count, "validation_valid_targets": val_count,
               "peak_vram_bytes": torch.cuda.max_memory_allocated() if train.device != "cpu" else 0}
        with (directory / "history.jsonl").open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(row)+"\n")
        print(json.dumps(row), flush=True)
        completed = epoch + 1
        if seconds_limit is not None and time.perf_counter() - run_started >= seconds_limit:
            budget_exhausted = True
            break
    return {"best_epoch": best_epoch, "best_val_l1_scaled": best,
            "epochs_completed": completed, "budget_exhausted": budget_exhausted}


### `src/models/gru_forecaster.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/models/gru_forecaster.py
"""Independent GRU artifact inference over observed histories only."""
import json
from pathlib import Path

import numpy as np
import torch

from src.data.panel import Panel

from src.features.gru_inputs import GRUPreprocessor
from src.models.gru import GRUNet


class GRUForecaster:
    """Load one seed/variant; return unscaled predictions for all three horizons."""

    def __init__(self, directory: str | Path, device: str = "cpu") -> None:
        directory = Path(directory)
        self.metadata = json.loads((directory / "meta.json").read_text())
        self.preprocessor = GRUPreprocessor.load(directory / "preprocessor.npz")
        self.device = device
        self.variant = self.metadata["variant"]
        self.name = f"gru_{self.variant}_seed{self.metadata['seed']}"
        self.model = GRUNet(2 if self.variant == "A" else 8, **self.metadata["architecture"]).to(device)
        state = torch.load(directory / "best.pt", map_location=device, weights_only=True)
        self.model.load_state_dict(state["model"])
        self.model.eval()

    def predict_all(self, history: np.ndarray, origin: int, batch_size: int = 256) -> np.ndarray:
        """Return (beam,3) forecasts using exactly the observed 168-hour window."""
        if history.shape != (168, len(self.preprocessor.beam_ids), 4) or origin < self.preprocessor.fit_end:
            raise ValueError("History shape or origin is incompatible with the fitted model")
        sequence = self.preprocessor.transform(history, self.variant).transpose(1, 0, 2)
        static = self.preprocessor.static_at(np.full(len(sequence), origin), np.arange(len(sequence)))
        predictions = []
        with torch.no_grad():
            for start in range(0, len(sequence), batch_size):
                x = torch.as_tensor(sequence[start:start+batch_size], device=self.device)
                s = torch.as_tensor(static[start:start+batch_size], device=self.device)
                predictions.append(self.model(x, s).cpu().numpy())
        scaled = np.concatenate(predictions)
        return scaled * self.preprocessor.scale[0] + self.preprocessor.mean[0]

    def predict_at(self, history: np.ndarray, origin: int, horizon: int) -> np.ndarray:
        """Select one requested horizon with absolute origin phase."""
        if horizon not in (1, 6, 24):
            raise ValueError("Unsupported horizon")
        return self.predict_all(history, origin)[:, (1, 6, 24).index(horizon)]

    def predict_origins(self, panel: Panel, origins: np.ndarray, batch_size: int = 256) -> np.ndarray:
        """Batch replay over all beams, returning (origin,beam,horizon) without future inputs."""
        origins = np.asarray(origins)
        if (origins.ndim != 1 or not origins.size or not np.issubdtype(origins.dtype, np.integer)
                or np.any(np.diff(origins) <= 0)
                or origins.min() < max(167, self.preprocessor.fit_end)
                or origins.max() >= len(panel.time)):
            raise ValueError("Origins must have sufficient observed history and valid indices")
        if panel.beam_ids != self.preprocessor.beam_ids:
            raise ValueError("Beam order or forecast origins mismatch")
        values = torch.as_tensor(self.preprocessor.transform(panel.values[:origins.max()+1],
                                                              self.variant), device=self.device)
        beams = np.arange(len(panel.beam_ids))
        pairs = np.array(np.meshgrid(origins, beams, indexing="ij")).reshape(2, -1).T
        offsets = torch.arange(-167, 1, device=self.device)
        result = []
        with torch.no_grad():
            for start in range(0, len(pairs), batch_size):
                batch = pairs[start:start+batch_size]
                t = torch.as_tensor(batch[:, 0], device=self.device)
                b = torch.as_tensor(batch[:, 1], device=self.device)
                history = values[t[:, None] + offsets, b[:, None]]
                static = torch.as_tensor(self.preprocessor.static_at(batch[:, 0], batch[:, 1]),
                                         device=self.device)
                result.append(self.model(history, static).cpu().numpy())
        scaled = np.concatenate(result).reshape(len(origins), len(beams), 3)
        return scaled * self.preprocessor.scale[0] + self.preprocessor.mean[0]

    def predict(self, history: np.ndarray, horizon: int) -> np.ndarray:
        raise ValueError("GRU requires absolute origin context via predict_at")


## 5. Orchestration và validation metrics

CLI dưới đây nối các module nhỏ với config. `train_gru` từ chối local training.
Validation replay dùng toàn bộ beam và cùng origin sets với baselines/LightGBM.
WAPE zero denominator trả NaN có cờ; zero FP là prediction > 0 khi actual == 0, không phải incident FPR.


### `src/evaluation/metrics.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/evaluation/metrics.py
"""Finite-pair metrics with explicit exclusion counts and undefined-result flags."""
from dataclasses import dataclass

import numpy as np


@dataclass(frozen=True)
class MetricResult:
    """Metric value and observation coverage; undefined values use NaN plus a flag."""

    value: float
    n_valid: int
    n_excluded: int
    flags: tuple[str, ...] = ()


def _pairs(actual: np.ndarray, predicted: np.ndarray) -> tuple[np.ndarray, np.ndarray, int]:
    actual, predicted = np.asarray(actual, dtype=float), np.asarray(predicted, dtype=float)
    if actual.shape != predicted.shape:
        raise ValueError("Actual and prediction shapes must match")
    valid = np.isfinite(actual) & np.isfinite(predicted)
    return actual[valid], predicted[valid], int(actual.size - valid.sum())


def _aggregate(actual: np.ndarray, predicted: np.ndarray, mode: str) -> MetricResult:
    a, p, excluded = _pairs(actual, predicted)
    if not a.size:
        return MetricResult(float("nan"), 0, excluded, ("no_valid_pairs",))
    if mode == "wape":
        denominator = np.abs(a).sum()
        if denominator == 0:
            return MetricResult(float("nan"), len(a), excluded, ("zero_denominator",))
        value = np.abs(a - p).sum() / denominator
    elif mode == "rmse":
        value = np.sqrt(np.mean((a - p) ** 2))
    else:
        value = np.abs(a - p).mean()
    return MetricResult(float(value), len(a), excluded)


def mae(actual: np.ndarray, predicted: np.ndarray) -> MetricResult:
    """Return mean absolute error over finite pairs."""
    return _aggregate(actual, predicted, "mae")


def rmse(actual: np.ndarray, predicted: np.ndarray) -> MetricResult:
    """Return root mean squared error over finite pairs."""
    return _aggregate(actual, predicted, "rmse")


def wape(actual: np.ndarray, predicted: np.ndarray) -> MetricResult:
    """Return sum absolute error / sum absolute actual (ratio, not percent)."""
    return _aggregate(actual, predicted, "wape")


def _macro(actual: np.ndarray, predicted: np.ndarray, metric) -> MetricResult:
    a, p = np.asarray(actual), np.asarray(predicted)
    if a.ndim != 2 or a.shape != p.shape:
        raise ValueError("Macro metrics require matching (origin, beam) arrays")
    results = [metric(a[:, i], p[:, i]) for i in range(a.shape[1])]
    values = [r.value for r in results if np.isfinite(r.value)]
    excluded_beams = len(results) - len(values)
    flags = (f"excluded_beams={excluded_beams}",) if excluded_beams else ()
    if not values:
        flags += ("no_valid_beams",)
    return MetricResult(float(np.mean(values)) if values else float("nan"),
                        sum(r.n_valid for r in results), sum(r.n_excluded for r in results), flags)


def macro_wape(actual: np.ndarray, predicted: np.ndarray) -> MetricResult:
    """Average defined per-beam WAPE, equally weighted; flag excluded beams."""
    return _macro(actual, predicted, wape)


def macro_mae(actual: np.ndarray, predicted: np.ndarray) -> MetricResult:
    """Average defined per-beam MAE, equally weighted."""
    return _macro(actual, predicted, mae)


def active_only_mae(actual: np.ndarray, predicted: np.ndarray) -> MetricResult:
    """Return MAE where actual > 0; excluded count includes inactive observations."""
    a, p, excluded = _pairs(actual, predicted)
    active = a > 0
    r = mae(a[active], p[active])
    return MetricResult(r.value, r.n_valid, excluded + int((~active).sum()), r.flags)


def zero_fp_rate(actual: np.ndarray, predicted: np.ndarray, epsilon: float = 0) -> MetricResult:
    """Fraction of actual-zero finite pairs predicted above epsilon (default strictly > 0)."""
    if not np.isfinite(epsilon) or epsilon < 0:
        raise ValueError("epsilon must be finite and nonnegative")
    a, p, excluded = _pairs(actual, predicted)
    zero = a == 0
    count = int(zero.sum())
    return MetricResult(float(np.mean(p[zero] > epsilon)) if count else float("nan"),
                        count, excluded + int((~zero).sum()),
                        () if count else ("no_zero_observations",))


def assign_load_tiers(fit_target: np.ndarray) -> np.ndarray:
    """Fit per-beam load tertiles on training-only (time, beam) target values.

    Equal means stay in the same tier. All-missing beams are Unknown; tied
    boundaries can leave a tier empty, rather than split identical loads arbitrarily.
    """
    values = np.asarray(fit_target, dtype=float)
    if values.ndim != 2 or not values.shape[0]:
        raise ValueError("Expected nonempty (fit_time, beam) array")
    valid = np.isfinite(values)
    counts = valid.sum(axis=0)
    means = np.divide(np.where(valid, values, 0).sum(0), counts,
                      out=np.full(values.shape[1], np.nan), where=counts > 0)
    tiers = np.full(values.shape[1], "Unknown", dtype="U7")
    observed = np.isfinite(means)
    if observed.any():
        cuts = np.quantile(means[observed], [1 / 3, 2 / 3])
        tiers[observed] = np.array(["Low", "Mid", "High"])[
            np.searchsorted(cuts, means[observed], side="left")]
    return tiers


### `src/evaluation/evaluator.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile src/evaluation/evaluator.py
"""Fixed-model rolling-origin evaluator with history-only predictor inputs."""
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Protocol

import numpy as np

from src.data.panel import Panel
from src.evaluation import metrics


class Forecaster(Protocol):
    """Predict one DLPRB value per beam from observed history only."""

    name: str

    def predict(self, history: np.ndarray, horizon: int) -> np.ndarray: ...


@dataclass
class ForecastRecord:
    """Arrays use (origin, beam), with explicit target indices and model identity."""

    model: str
    horizon: int
    origins: np.ndarray
    beam_ids: tuple[str, ...]
    actual: np.ndarray
    predicted: np.ndarray


def history_at(panel: Panel, origin: int, lookback: int = 168) -> np.ndarray:
    """Return an independent read-only window ending at origin; no future rows."""
    if not isinstance(origin, (int, np.integer)) or isinstance(origin, bool):
        raise ValueError("origin must be an integer")
    if lookback < 1 or origin < lookback - 1 or origin >= len(panel.time):
        raise ValueError("Origin has insufficient history or is out of bounds")
    history = panel.values[origin - lookback + 1:origin + 1].copy()
    history.flags.writeable = False
    return history


def evaluate(forecaster: Forecaster, panel: Panel, origins: np.ndarray,
             horizon: int, lookback: int = 168) -> ForecastRecord:
    """Evaluate fixed-model replay; actuals observed through each origin update history."""
    if not np.array_equal(panel.time, np.arange(len(panel.time))):
        raise ValueError("Evaluation requires contiguous relative hourly indices from zero")
    if hasattr(forecaster, "static") and forecaster.static.beam_ids != panel.beam_ids:
        raise ValueError("Model and panel beam ordering differ")
    if not isinstance(horizon, (int, np.integer)) or isinstance(horizon, bool) or horizon < 1:
        raise ValueError("horizon must be a positive integer")
    origins = np.asarray(origins)
    if origins.ndim != 1 or not np.issubdtype(origins.dtype, np.integer) or not origins.size:
        raise ValueError("Expected a nonempty integer origin vector")
    if np.any(np.diff(origins) <= 0) or origins[-1] + horizon >= len(panel.time):
        raise ValueError("Origins must increase uniquely and targets must be in bounds")
    forecasts = []
    for origin in origins:
        history = history_at(panel, int(origin), lookback)
        if hasattr(forecaster, "predict_at"):
            # Time context supplies phase only; observed inputs remain history-only.
            prediction = forecaster.predict_at(history, int(origin), horizon)
        else:
            prediction = forecaster.predict(history, horizon)
        prediction = np.asarray(prediction, dtype=float)
        if prediction.shape != (len(panel.beam_ids),) or np.isinf(prediction).any():
            raise ValueError("Forecast must be one value per beam, finite or NaN")
        forecasts.append(prediction.copy())
    return ForecastRecord(forecaster.name, horizon, origins.copy(), panel.beam_ids,
                          panel.values[origins + horizon, :, 0].copy(), np.stack(forecasts))


def to_benchmark_rows(record: ForecastRecord, load_tiers: np.ndarray | None = None) -> list[dict]:
    """Summarize pooled and per-load-tier metrics with missingness/coverage counts."""
    groups = {"All": np.ones(len(record.beam_ids), dtype=bool)}
    if load_tiers is not None:
        if np.asarray(load_tiers).shape != (len(record.beam_ids),):
            raise ValueError("Expected one load tier per beam")
        groups.update({tier: load_tiers == tier for tier in ("Low", "Mid", "High", "Unknown")
                       if np.any(load_tiers == tier)})
    functions = {"mae": metrics.mae, "rmse": metrics.rmse, "wape": metrics.wape,
                 "macro_wape": metrics.macro_wape, "macro_mae": metrics.macro_mae,
                 "active_only_mae": metrics.active_only_mae, "zero_fp_rate": metrics.zero_fp_rate}
    rows = []
    for tier, mask in groups.items():
        a, p = record.actual[:, mask], record.predicted[:, mask]
        valid = np.isfinite(a) & np.isfinite(p)
        row = {"model": record.model, "horizon": record.horizon, "load_tier": tier,
               "n_origins": len(record.origins), "n_beams": int(mask.sum()),
               "n_total": a.size, "n_valid": int(valid.sum()),
               "n_excluded": int((~valid).sum()), "n_missing_target": int(np.isnan(a).sum()),
               "n_missing_prediction": int(np.isnan(p).sum()),
               "n_excluded_beams": int((~valid.any(axis=0)).sum()),
               "n_active": int(((a > 0) & valid).sum()),
               "n_zero": int(((a == 0) & valid).sum())}
        for name, function in functions.items():
            result = function(a, p)
            row[name] = result.value
            row[f"{name}_flags"] = "|".join(result.flags)
        rows.append(row)
    return rows


def save_forecasts(records: list[ForecastRecord], path: str | Path,
                    metadata: dict | None = None) -> None:
    """Save variable-length horizon records in a pickle-free compressed NPZ."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    arrays = {}
    index = []
    for i, record in enumerate(records):
        prefix = f"record_{i}"
        index.append({"prefix": prefix, "model": record.model, "horizon": record.horizon})
        for name, value in {"origins": record.origins, "targets": record.origins + record.horizon,
                            "beam_ids": np.array(record.beam_ids), "actual": record.actual,
                            "predicted": record.predicted}.items():
            arrays[f"{prefix}_{name}"] = value
    arrays["metadata"] = np.array(json.dumps({"records": index, **(metadata or {})}))
    np.savez_compressed(path, **arrays)


### `scripts/prepare_data.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile scripts/prepare_data.py
"""Run Week 1 ingestion and audit: python -m scripts.prepare_data --step all."""
import argparse
import json
import logging
from pathlib import Path

import matplotlib
import numpy as np
import yaml

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.data.audit import (
    audit_beams, audit_duplicates, audit_missing, audit_range, audit_zeros,
    check_temporal_boundary,
)
from src.data.io import build_manifest, download_dataset
from src.data.panel import Panel, build_panel, load_panel, save_panel
from src.data.schema import KPIS, SEGMENTS, csv_path
from src.data.splits import build_split_config
from src.utils.config import load_config
from src.utils.metadata import collect_run_metadata
from src.utils.seed import set_seed


def write_json(path: Path, value: dict) -> None:
    """Write a UTF-8 JSON report."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, allow_nan=False), encoding="utf-8")


def plot_audit(panel: Panel, reports: Path) -> None:
    """Save distribution, missing/zero profile, series and phase figures."""
    folder = reports / "figures"
    folder.mkdir(exist_ok=True)
    target = panel.values[:, :, 0]
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    observed = target[np.isfinite(target)]
    axes[0, 0].hist(observed, bins=60, log=True)
    axes[0, 0].set_title("DLPRB usage distribution (log count)")
    axes[0, 1].hist((target == 0).sum(0) / np.maximum(np.isfinite(target).sum(0), 1), bins=40)
    axes[0, 1].set_title("Per-beam zero ratio (observed denominator)")
    axes[1, 0].plot(panel.time, target[:, :3])
    axes[1, 0].set_title("Sample beams: DLPRB usage")
    for period in (24, 168):
        profile = [float(np.nanmean(target[panel.time % period == phase]))
                   if np.isfinite(target[panel.time % period == phase]).any() else np.nan
                   for phase in range(period)]
        axes[1, 1].plot(range(period), profile, label=f"{period}h phase")
    axes[1, 1].legend()
    axes[1, 1].set_title("Phase index, not calendar time")
    fig.tight_layout()
    fig.savefig(folder / "data_audit.png", dpi=140)
    plt.close(fig)


def main() -> None:
    """Execute download, panel construction, audit, or all stages."""
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", default="configs/data.yaml")
    parser.add_argument("--step", choices=["download", "panel", "audit", "all"], default="all")
    args = parser.parse_args()
    config = load_config(args.config)
    raw = Path(config["raw_dir"])
    processed = Path(config["processed_dir"])
    reports = Path(config["reports_dir"])
    reports.mkdir(parents=True, exist_ok=True)
    processed.mkdir(parents=True, exist_ok=True)
    logging.basicConfig(level=logging.INFO, handlers=[logging.StreamHandler(),
                        logging.FileHandler(reports / "prepare_data.log", encoding="utf-8")])
    logging.getLogger("httpx").setLevel(logging.WARNING)
    set_seed(config["seed"])
    logging.info("Starting step=%s revision=%s", args.step, config["revision"])
    if args.step in ("download", "all"):
        download_dataset(config["repo_id"], config["revision"], raw)
        write_json(reports / "data_manifest.json", build_manifest(raw))
    if args.step in ("panel", "all"):
        manifest = build_manifest(raw)
        write_json(reports / "data_manifest.json", manifest)
        metadata = {"dataset_revision": manifest["revision"], "license": manifest["license"]}
        panels = {segment: build_panel({k: raw / csv_path(k, segment) for k in KPIS},
                                       {**metadata, "segment": segment}) for segment in SEGMENTS}
        boundary = check_temporal_boundary(panels["train"], panels["week6"],
                    verified=config["continuity_verified"], evidence=config["continuity_evidence"])
        panel = panels["train"]
        if boundary["continuity_verified"]:
            panel = Panel(np.concatenate([panel.values, panels["week6"].values]),
                          np.arange(1008), panel.beam_ids,
                          {**metadata, "segment": "train+week6", "boundary": boundary})
        if not np.array_equal(panel.time, np.arange(len(panel.time))):
            raise ValueError("Main panel requires contiguous hourly indices starting at zero")
        split = build_split_config(len(panel.time), boundary["continuity_verified"])
        save_panel(panel, processed / "panel.npz")
        save_panel(panels["week6"], processed / "week6.npz")
        save_panel(panels["week11"], processed / "week11.npz")
        write_json(processed / "split.json", split)
        evaluation_path = Path(config["evaluation_config"])
        evaluation_path.parent.mkdir(parents=True, exist_ok=True)
        evaluation_path.write_text(yaml.safe_dump({
            "lookback": split["lookback"], "horizons": split["horizons"],
            "evaluation_stride": 1, "split_mode": split["mode"],
            "fit": split["fit"], "validation": split["validation"],
            "holdout": split["holdout"],
        }, sort_keys=False), encoding="utf-8")
        write_json(reports / "temporal_boundary.json", boundary)
        (reports / "split_decision.md").write_text(
            f"# Temporal split decision\n\nMode: **{split['mode']}**.\n\n"
            f"{boundary['reason']}.\n\nEvidence: {boundary['evidence'] or 'Not approved'}.\n\n"
            "Source: dataset_card.md at pinned revision. Source indices reset for week 6; "
            "no calendar timestamps are supplied. No profile correlation is used as evidence.\n\n"
            f"Inclusive bounds:\n```json\n{json.dumps(split, indent=2)}\n```\n"
            "Week 11 is separate. Model selection uses validation only.\n", encoding="utf-8")
    if args.step in ("audit", "all"):
        panel = load_panel(processed / "panel.npz")
        audit = {"shape": list(panel.values.shape), "missing": audit_missing(panel),
                 "zeros": audit_zeros(panel), "range": audit_range(panel),
                 "duplicates": audit_duplicates(panel), "beams": audit_beams(panel)}
        write_json(reports / "data_audit.json", audit)
        (reports / "data_audit.md").write_text(
            "# Data audit\n\nDescriptive audit of the main panel; no imputation or filtering.\n\n"
            "NaN is preserved; zero is a valid observation. DLPRB unit remains unverified; "
            "use **DLPRB usage**. Active means value > 0 among observed values.\n\n"
            f"```json\n{json.dumps({k: v for k, v in audit.items() if k != 'missing'}, indent=2)}\n```\n"
            f"\nMissing ratios per KPI: {audit['missing']['per_kpi']}\n\n"
            "Full per-beam missing ratios: data_audit.json.\n\n"
            "![Audit figures](figures/data_audit.png)\n", encoding="utf-8")
        plot_audit(panel, reports)
    write_json(reports / "run_metadata.json", collect_run_metadata(config))
    logging.info("Completed step=%s", args.step)


if __name__ == "__main__":
    main()


### `scripts/train_gru.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile scripts/train_gru.py
"""GPU-only GRU-A/B training with a shared, training-stratified beam sample."""
import argparse
import json
import os
from pathlib import Path

import numpy as np
import torch

from src.evaluation.origins import evaluation_origins, train_origins_joint
from src.evaluation.validation import validation_inputs
from src.features.beam_sampling import sample_beams
from src.features.gru_inputs import GRUBatches, GRUPreprocessor
from src.features.static_stats import compute_static
from src.models.gru import GRUNet
from src.models.gru_train import train_gru
from src.utils.config import load_config
from src.utils.metadata import collect_run_metadata
from src.utils.seed import set_seed
from src.utils.runtime import require_cloud_training


def main() -> None:
    """Train one requested variant/seed; use --pilot for exactly one measured epoch."""
    require_cloud_training()
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--variant", choices=["A", "B"], required=True)
    parser.add_argument("--seed", type=int, default=17)
    parser.add_argument("--pilot", action="store_true")
    parser.add_argument("--resume", action="store_true")
    parser.add_argument("--seconds-limit", type=float, default=None)
    parser.add_argument("--config", default="configs/gru.yaml")
    parser.add_argument("--data-config", default="configs/data.yaml")
    args = parser.parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("GRU training requires a CUDA GPU; CPU training is disabled")
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    torch.set_num_threads(2)
    set_seed(args.seed)
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    data_config, config = load_config(args.data_config), load_config(args.config)
    if args.seed not in config["seeds"] or args.variant not in config["variants"]:
        raise ValueError("Requested run is outside the frozen experiment list")
    panel, split = validation_inputs(data_config)
    preprocessor = GRUPreprocessor.fit(panel, split["fit"][1], args.variant)
    selected = sample_beams(compute_static(panel, split["fit"][1]), config["sample_beams"],
                             config["sample_seed"])
    reports = Path(data_config["reports_dir"])
    reports.mkdir(parents=True, exist_ok=True)
    sample_manifest = {"fit_end": split["fit"][1], "seed": config["sample_seed"],
                       "indices": selected.tolist(), "beam_ids": [panel.beam_ids[i] for i in selected],
                       "method": "proportional joint tertiles of fit mean/std/active ratio"}
    (reports / "gru_sample_manifest.json").write_text(json.dumps(sample_manifest, indent=2))
    train_origins = train_origins_joint(split["fit"][1], stride=config["training_stride"])
    val_origins = evaluation_origins(split["fit"][1], split["validation"][1], 24)
    train = GRUBatches(panel, preprocessor, args.variant, train_origins, selected, "cuda")
    validation = GRUBatches(panel, preprocessor, args.variant, val_origins,
                            np.arange(len(panel.beam_ids)), "cuda")
    directory = Path(config["artifact_dir"]) / args.variant / f"seed{args.seed}"
    directory.mkdir(parents=True, exist_ok=True)
    preprocessor.save(directory / "preprocessor.npz")
    if (directory / "last.pt").exists() and not args.resume:
        raise ValueError("Existing run requires --resume; refusing to overwrite checkpoint history")
    model = GRUNet(2 if args.variant == "A" else 8, config["hidden_size"],
                    config["num_layers"], config["dropout"])
    training = dict(config)
    if args.pilot:
        training["max_epochs"] = 1
    identity = {"variant": args.variant, "dataset_revision": data_config["revision"],
                "schema": "gru-global-kpi-phase4-v2",
                "fit_end": split["fit"][1], "validation_end": split["validation"][1],
                "beam_sample": selected.tolist()}
    torch.cuda.reset_peak_memory_stats()
    summary = train_gru(model, train, validation, training, directory, args.seed, args.resume,
                        run_identity=identity, seconds_limit=args.seconds_limit)
    meta = {**collect_run_metadata({**data_config, "gru": config}), **summary,
            "variant": args.variant, "seed": args.seed, "architecture": {
                k: config[k] for k in ("hidden_size", "num_layers", "dropout")},
            "fit_end": split["fit"][1], "train_samples": train.n_samples,
            "preprocessing_schema": "gru-global-kpi-phase4-v2", "phase_features": 4,
            "validation_joint_samples": validation.n_samples, "pilot": args.pilot,
            "torch_version": torch.__version__, "gpu": torch.cuda.get_device_name(0)}
    (directory / "meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")


if __name__ == "__main__":
    main()


### `scripts/evaluate_gru.py`

Sửa cell này rồi chạy lại trước các cell sử dụng module. Nếu đã import module trong phiên, restart kernel hoặc reload module để dùng code mới.


In [ ]:
%%writefile scripts/evaluate_gru.py
"""Validation-only GRU replay, seed summaries and A/B ablation."""
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from src.evaluation.evaluator import ForecastRecord, save_forecasts, to_benchmark_rows
from src.evaluation.metrics import assign_load_tiers
from src.evaluation.origins import evaluation_origins
from src.evaluation.validation import validation_inputs
from src.models.gru_forecaster import GRUForecaster
from src.utils.config import load_config
from src.utils.metadata import collect_run_metadata


def evaluate_gru(data_config_path: str = "configs/data.yaml", config_path: str = "configs/gru.yaml") -> None:
    """Reload every declared run and compare full-beam validation with LightGBM/baselines."""
    data_config, config = load_config(data_config_path), load_config(config_path)
    panel, split = validation_inputs(data_config)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.set_num_threads(2)
    if device == "cpu" and len(panel.beam_ids) > 32:
        raise RuntimeError("Full GRU replay requires GPU; CPU replay reserved for small tests")
    reports = Path(data_config["reports_dir"])
    tiers = assign_load_tiers(panel.values[:split["fit"][1]+1, :, 0])
    origin_sets = {h: evaluation_origins(split["fit"][1], split["validation"][1], h)
                   for h in (1, 6, 24)}
    records, rows, failures = [], [], []
    for variant in config["variants"]:
        for seed in config["seeds"]:
            directory = Path(config["artifact_dir"]) / variant / f"seed{seed}"
            model = GRUForecaster(directory, device)
            if (model.metadata["dataset_revision"] != data_config["revision"]
                    or model.preprocessor.fit_end != split["fit"][1]):
                raise ValueError("GRU artifact revision/fit boundary mismatch")
            all_predictions = model.predict_origins(panel, origin_sets[1], config["batch_size"])
            t = int(origin_sets[1][0])
            np.testing.assert_allclose(all_predictions[0], model.predict_all(panel.values[t-167:t+1], t),
                                       rtol=1e-4, atol=1e-5)
            for k, h in enumerate((1, 6, 24)):
                origins = origin_sets[h]
                record = ForecastRecord(model.name, h, origins, panel.beam_ids,
                    panel.values[origins+h, :, 0].copy(), all_predictions[:len(origins), :, k])
                records.append(record)
                current = to_benchmark_rows(record, tiers)
                for row in current:
                    row.update(variant=variant, seed=seed, best_epoch=model.metadata["best_epoch"])
                rows.extend(current)
                absolute = np.abs(record.actual-record.predicted)
                valid = np.flatnonzero(np.isfinite(absolute))
                for flat in valid[np.argsort(absolute.reshape(-1)[valid])[-3:][::-1]]:
                    oi, bi = np.unravel_index(flat, absolute.shape)
                    failures.append({"model": model.name, "horizon": h,
                        "origin": int(origins[oi]), "beam_id": panel.beam_ids[bi],
                        "actual": record.actual[oi, bi], "prediction": record.predicted[oi, bi],
                        "absolute_error": absolute[oi, bi]})
            print(f"GRU {variant} seed{seed}: full-beam validation replay passed", flush=True)
    frame = pd.DataFrame(rows)
    frame.to_csv(reports/"gru_benchmark_val.csv", index=False)
    pooled = frame[frame.load_tier == "All"]
    summary = pooled.groupby(["variant", "horizon"]).agg(
        mae_mean=("mae", "mean"), mae_sd=("mae", "std"), seeds=("seed", "nunique"),
        active_only_mae_mean=("active_only_mae", "mean"),
        zero_fp_rate_mean=("zero_fp_rate", "mean")).reset_index()
    summary.to_csv(reports/"ablation_study.csv", index=False)
    failure_frame = pd.DataFrame(failures)
    failure_frame.to_csv(reports/"gru_failure_examples.csv", index=False)
    baseline = pd.read_csv(reports/"baseline_benchmark_val.csv")
    best = baseline[baseline.load_tier == "All"].groupby("horizon").mae.min()
    summary["best_baseline_mae"] = summary.horizon.map(best)
    summary["skill_vs_baseline"] = 1-summary.mae_mean/summary.best_baseline_mae
    lightgbm = pd.read_csv(reports/"lightgbm_skill_val.csv")
    selected = load_config("configs/lightgbm.yaml")["selected_config"]
    lgb = lightgbm[lightgbm.model == f"lightgbm_{selected}"].set_index("horizon").mae
    summary["lightgbm_mae"] = summary.horizon.map(lgb)
    summary["skill_vs_lightgbm"] = 1-summary.mae_mean/summary.lightgbm_mae
    summary.to_csv(reports/"gru_comparison_val.csv", index=False)
    metadata = collect_run_metadata({**data_config, "gru": config, "segment": "val"})
    (reports/"gru_run_metadata.json").write_text(json.dumps(metadata, indent=2))
    save_forecasts(records, reports/"forecasts/gru_val.npz", metadata)
    lines = ["# GRU validation error analysis", "", "No holdout evaluated. Predictions are un-clipped.",
             "Mean +/- sample SD across seeds is descriptive only; two seeds do not establish stability.",
             "One-seed SD is undefined. Early stopping uses joint-horizon validation origins; final replay",
             "uses the same per-horizon full-beam origin sets as LightGBM and baselines.", ""]
    for title, table in (("A/B and model comparison", summary), ("Load/active/zero metrics", frame),
                          ("Largest validation errors (not incident labels)", failure_frame)):
        lines += [f"## {title}", "", "```csv", table.to_csv(index=False).strip(), "```", ""]
    (reports/"gru_error_analysis.md").write_text("\n".join(lines), encoding="utf-8")


## 6. Cấu hình có thể review trước khi chạy

Chỉ hai variants A/B và seeds 17/42; không có grid search ẩn.


In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=69"]
build-backend = "setuptools.build_meta"

[project]
name = "5g-network-intelligence"
version = "0.1.0"
description = "A reproducible 5G beam-level KPI forecasting and historical capacity-risk monitoring system using LightGBM, GRU, FastAPI, Streamlit, and Docker."
requires-python = ">=3.12"
dependencies = ["numpy>=2,<3", "pandas>=2.2,<4", "PyYAML>=6,<7", "huggingface-hub>=0.27,<2", "matplotlib>=3.9,<4"]

[project.optional-dependencies]
dev = ["pytest>=8,<10", "ruff>=0.9,<1"]
ml = ["lightgbm>=4.6,<5", "scikit-learn>=1.6,<2"]
neural = ["torch>=2.6,<3"]

[tool.setuptools.packages.find]
include = ["src*"]

[tool.pytest.ini_options]
pythonpath = ["."]
testpaths = ["tests"]
addopts = "--basetemp=.pytest_tmp"
markers = ["cloud_training: model fitting tests allowed only on hosted Colab/Kaggle"]

[tool.ruff]
line-length = 100
target-version = "py312"
extend-exclude = [".uv-cache", ".venv", ".pytest_tmp"]

[tool.ruff.lint]
select = ["E4", "E7", "E9", "F"]


In [ ]:
%%writefile configs/data.yaml
repo_id: netop/gotsf-ds
revision: c9ecaf153354b2cc415c64750a966c47c7f3f6ac
raw_dir: data/raw
processed_dir: data/processed
reports_dir: reports
evaluation_config: configs/evaluation.yaml
seed: 17
# Dataset card calls week 6 the immediate future, but its Time index resets.
# Conservative fallback until an explicit continuity review is recorded.
continuity_verified: false
continuity_evidence: ''


In [ ]:
%%writefile configs/evaluation.yaml
lookback: 168
horizons:
- 1
- 6
- 24
evaluation_stride: 1
split_mode: fallback
fit:
- 0
- 503
validation:
- 504
- 671
holdout:
- 672
- 839


In [ ]:
%%writefile configs/gru.yaml
hidden_size: 64
num_layers: 2
dropout: 0.1
learning_rate: 0.001
batch_size: 256
max_epochs: 25
early_stopping_patience: 5
gradient_clip: 1.0
training_stride: 3
sample_beams: 600
sample_seed: 17
seeds: [17, 42]
variants: [A, B]
artifact_dir: artifacts/gru_w5
gpu_budget_seconds: 14400


In [ ]:
%%writefile configs/lightgbm.yaml
seed: 17
training_stride: 2
num_threads: 4
artifact_dir: artifacts/lgbm_w5
selected_config: c1
configs:
  c1:
    objective: regression_l1
    num_leaves: 31
    learning_rate: 0.05
    n_estimators: 500
    early_stopping_rounds: 50
  c2:
    objective: regression
    num_leaves: 31
    learning_rate: 0.05
    n_estimators: 500
    early_stopping_rounds: 50


### Mốc baseline và LightGBM đã đo

Các CSV rút gọn này chỉ dùng so sánh metric khi replay GRU; không được dùng làm training targets.


In [ ]:
%%writefile reports/baseline_benchmark_val.csv
model,horizon,load_tier,mae
persistence,1,All,0.29774736361494325
persistence,6,All,0.49691991791829715
persistence,24,All,0.3320654789463364
seasonal_naive_24,1,All,0.3299414844066566
seasonal_naive_24,6,All,0.33269880428002646
seasonal_naive_24,24,All,0.3320654789463364
seasonal_naive_168,1,All,0.3358355360302166
seasonal_naive_168,6,All,0.33869466549698707
seasonal_naive_168,24,All,0.33667361269701107


In [ ]:
%%writefile reports/lightgbm_skill_val.csv
model,horizon,mae
lightgbm_c1,1,0.23063451146783154
lightgbm_c1,6,0.25347289601808365
lightgbm_c1,24,0.2512464622398822
lightgbm_c2,1,0.23886068678595976
lightgbm_c2,6,0.26495300379453174
lightgbm_c2,24,0.2633524960304302


### Bước 1 — Dependencies và dữ liệu

Bật riêng từng cờ khi cần. Tải dữ liệu public ở revision đã pin; bước này không train.


In [ ]:
if RUN_INSTALL:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev,ml,neural]'], check=True)
if RUN_PREPARE_DATA:
    subprocess.run([sys.executable, '-m', 'scripts.prepare_data', '--step', 'all'], check=True)
else:
    print('Review mode: bỏ qua download/prepare data.')


### Bước 2 — Debug một beam/origin trước khi train

Kiểm tra shape, missing masks và target alignment. Preprocessor chỉ dùng fit segment.


In [ ]:
if DEBUG_DATA:
    import numpy as np
    from src.utils.config import load_config
    from src.evaluation.validation import validation_inputs
    from src.features.gru_inputs import GRUPreprocessor, GRUBatches
    panel, split = validation_inputs(load_config('configs/data.yaml'))
    prep = GRUPreprocessor.fit(panel, split['fit'][1], VARIANT)
    batch = GRUBatches(panel, prep, VARIANT, np.array([DEBUG_ORIGIN]), np.array([DEBUG_BEAM]))
    x, phase4, target3 = next(batch.batches(1))
    print('sequence:', tuple(x.shape), 'static:', tuple(phase4.shape), 'target:', tuple(target3.shape))
    print('target indices:', DEBUG_ORIGIN + np.array([1, 6, 24]))
    print('targets scaled:', target3)
else:
    print('Bật RUN_PREPARE_DATA và DEBUG_DATA để inspect dữ liệu.')


### Bước 3 — Training có checkpoint

Chỉ chạy nếu chủ động bật RUN_TRAINING và runtime có CUDA.
Giữ RUN_ALL_EXPERIMENTS=False để debug một run; bật True để chạy A/B × seeds 17/42.
Đặt artifact_dir in configs/gru.yaml trên Drive hoặc dùng Kaggle Output để checkpoint bền vững.


In [ ]:
if RUN_TRAINING:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Chọn GPU runtime trước khi training.')
    # Danh sách experiments hiển thị rõ; không có vòng hyperparameter search ẩn.
    experiments = [('A', 17), ('A', 42), ('B', 17), ('B', 42)] if RUN_ALL_EXPERIMENTS else [(VARIANT, SEED)]
    for variant, seed in experiments:
        command = [sys.executable, '-m', 'scripts.train_gru', '--variant', variant,
                   '--seed', str(seed), '--seconds-limit', '360']
        if RESUME:
            command.append('--resume')
        print('Running:', ' '.join(command))
        subprocess.run(command, check=True)
else:
    print('Review mode: không train lại. Các kết quả đầu notebook là runs đã hoàn tất.')


### Bước 4 — Learning curves và replay

Đọc loss theo epoch để debug divergence/early stopping. Replay cần đủ artifacts đã khai báo trong configs/gru.yaml.


In [ ]:
from src.utils.config import load_config
history_path = Path(load_config('configs/gru.yaml')['artifact_dir']) / VARIANT / f'seed{SEED}' / 'history.jsonl'
if history_path.exists():
    history = pd.read_json(history_path, lines=True)
    display(history)
    history.plot(x='epoch', y=['train_l1_scaled', 'val_l1_scaled'], title=f'GRU-{VARIANT}, seed {SEED}')
else:
    print('Checkpoint phase4: mở Week 5 GPU job, Version 1 → Output; review không train lại.')

if RUN_EVALUATION:
    from scripts.evaluate_gru import evaluate_gru
    evaluate_gru()
    display(pd.read_csv('reports/gru_comparison_val.csv'))
